In [0]:
BRONZE_PATH = "/Volumes/workspace/finance_analytics/finance_raw/bronze"

print(BRONZE_PATH)

In [0]:
from pyspark.sql import functions as F

# =========================================================
# CONFIGURATION
# =========================================================

BASE_PATH = "/Volumes/workspace/finance_analytics/finance_raw"

BRONZE_PATH = "/Volumes/workspace/finance_analytics/finance_raw/bronze"

# =========================================================
# LOAD RAW CUSTOMERS
# =========================================================

customers_raw_df = spark.read.parquet(
    f"{BASE_PATH}/customers"
)

print(f"Raw customers: {customers_raw_df.count():,}")

# =========================================================
# WRITE CUSTOMERS TO BRONZE AS DELTA
# =========================================================

customers_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}/customers")

print("Customers successfully loaded into Bronze.")

In [0]:
bronze_customers_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/customers"
)

print(f"Bronze customers: {bronze_customers_df.count():,}")

display(bronze_customers_df.limit(10))

bronze_customers_df.printSchema()

In [0]:
# =========================================================
# LOAD ALL REMAINING DATASETS INTO BRONZE
# =========================================================

datasets = [
    "products",
    "branches",
    "accounts",
    "transactions",
    "loans",
    "loan_payments"
]

for dataset in datasets:

    print(f"Loading {dataset}...")

    df = spark.read.parquet(
        f"{BASE_PATH}/{dataset}"
    )

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{BRONZE_PATH}/{dataset}")

    print(f"{dataset}: {df.count():,} rows loaded")

In [0]:
# =========================================================
# VERIFY ALL BRONZE DATASETS
# =========================================================

for dataset in [
    "customers",
    "products",
    "branches",
    "accounts",
    "transactions",
    "loans",
    "loan_payments"
]:

    df = spark.read.format("delta").load(
        f"{BRONZE_PATH}/{dataset}"
    )

    print(f"{dataset}: {df.count():,} rows")

In [0]:
# =========================================================
# BRONZE LAYER VALIDATION
# =========================================================

print("=== BRONZE LAYER VALIDATION ===")

bronze_counts = {}

for dataset in [
    "customers",
    "products",
    "branches",
    "accounts",
    "transactions",
    "loans",
    "loan_payments"
]:
    df = spark.read.format("delta").load(
        f"{BRONZE_PATH}/{dataset}"
    )

    bronze_counts[dataset] = df.count()

for dataset, count in bronze_counts.items():
    print(f"{dataset}: {count:,}")

print("\nBronze ingestion validation completed.")